In [1]:
import sys
print("当前运行的 Python 路径:", sys.executable)
!pip show langchain

当前运行的 Python 路径: d:\CodeRelated\codeService\anaconda3\envs\LangChainPy310\python.exe
Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: d:\coderelated\codeservice\anaconda3\envs\langchainpy310\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: langchain-tavily


# 1、ChatMessageHistory的使用


In [2]:
# 相关导入
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
)
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
)
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate,
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    FewShotPromptTemplate,
)
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains.llm import LLMChain

In [3]:


# 1、 ChatMessageHistory的实例化
history = InMemoryChatMessageHistory()
# 2、添加相关的消息进行存储
history.add_user_message("你好")

history.add_ai_message("很高兴认识你")

# 3、打印存储的消息
print(history.messages)

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [4]:
# 获取大模型
# 创建大模型
# 加载环境变量
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)


# 1.提供大模型
chat_model = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
)

In [5]:
history.add_user_message("请帮我计算1+2*3=？")
response = chat_model.invoke(history.messages)
print(type(history))
print(response.content)

<class 'langchain_core.chat_history.InMemoryChatMessageHistory'>
当然可以！我们来一步一步计算这个表达式：  
**1 + 2 * 3**

根据数学运算的优先级，**乘法**的优先级高于**加法**，所以先算乘法部分：

1. 计算乘法：  
   **2 * 3 = 6**

2. 然后进行加法：  
   **1 + 6 = 7**

所以，**1 + 2 * 3 = 7**。


# ConversationBufferMemory的使用

In [6]:
# 1.创建ConversationBufferMemory的实例化
memory = ConversationBufferMemory()

# 2. 存储相关的消息
# inputs对应的就是用户信息，outputs对应的就是ai消息
memory.save_context(
    inputs={"human": "你好，我叫小明"},
    outputs={"ai": "很高兴认识你"},
)
memory.save_context(
    inputs={"input": "帮我谁打一下1+2*3=？"},
    outputs={"output": "7"},
)

# 3. 获取存储的信息
# 返回消息列表的方式1：
print(memory.load_memory_variables({}))

# 返回消息列表的方式2：
print(memory.chat_memory.messages)

# 说明：返回的字典结构的key叫history。

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我谁打一下1+2*3=？\nAI: 7'}
[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我谁打一下1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


C:\Users\javis\AppData\Local\Temp\ipykernel_47372\3625462460.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [7]:
# 結合大模型、提示詞模板的使用

prompt_template = PromptTemplate.from_template(
    template="""
  你可以与人类对话。
  当前对话历史：{history}
  人类问题：{question}
  回复：
  """
)

# 提供memory实例
memory = ConversationBufferMemory()

# 提供chain
chain = LLMChain(llm=chat_model, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})

# 此时history的内容为空
print(response)

C:\Users\javis\AppData\Local\Temp\ipykernel_47372\1347048691.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=chat_model, prompt=prompt_template, memory=memory)


{'question': '你好，我的名字叫小明', 'history': '', 'text': '你好，小明！很高兴认识你。有什么我可以帮助你的吗？'}


In [8]:
prompt_template = PromptTemplate.from_template(
    template="""
  你可以与人类对话。
  当前对话历史：{history}
  人类问题：{question}
  回复：
  """
)

# 显示的设置memory的key的值
# 1.创建ConversationBufferMemory的实例化
memory = ConversationBufferMemory(memory_key="history")

# 2. 存储相关的消息
# inputs对应的就是用户信息，outputs对应的就是ai消息
memory.save_context(
    inputs={"human": "你好，我叫小明"},
    outputs={"ai": "很高兴认识你"},
)
memory.save_context(
    inputs={"input": "帮我谁打一下1+2*3=？"},
    outputs={"output": "7"},
)

# 3. 获取存储的信息
# 返回消息列表的方式1：
print(memory.load_memory_variables({}))

# 返回消息列表的方式2：
print(memory.chat_memory.messages)

# 说明：返回的字典结构的key叫history。

# 提供chain
chain = LLMChain(llm=chat_model, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})

# 此时history的内容为空
print(response)

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我谁打一下1+2*3=？\nAI: 7'}
[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我谁打一下1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
{'question': '你好，我的名字叫小明', 'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我谁打一下1+2*3=？\nAI: 7', 'text': '你好，小明！很高兴再次见到你。今天有什么我可以帮你的吗？'}


In [9]:
# 结合大模型使用提示词模板ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(
    template="""
  你可以与人类对话。
  当前对话历史：{history}
  人类问题：{question}
  回复：
  """
)

# 显示的设置memory的key的值
# 1.创建ConversationBufferMemory的实例化
# ==============================================================================
# 🧠 ConversationBufferMemory 实例化：创建一个“全量对话记忆管家”
#
# 作用：它会在内存中维护一个列表，原封不动地保存用户和AI的所有历史对话记录。
# 特点：“Buffer（缓冲）”意味着它不做任何摘要或裁剪，直接把所有历史记录拼成一段长文本。
# ==============================================================================
memory = ConversationBufferMemory(
    # 🌟 memory_key (极其重要！这是记忆和 Prompt 模板对接的“接头暗号”)
    #
    # 解释：当你把这段记忆塞给 LLMChain 时，LLMChain 需要把这段记忆“填”进提示词模板中。
    # 这里的 "history" 就是你要在 PromptTemplate 里留出的变量名（占位符）。
    #
    # 举例：
    # 如果你的 memory_key="history"
    # 那么你的 PromptTemplate 里面必须有一句类似这样的话：
    # "之前的聊天记录是：\n{history}\n现在请回答用户的新问题..."
    #
    # 警告：如果不写这个参数，LangChain 默认的 memory_key 也是 "history"。
    # 但如果你的模板里写的是 {chat_history}，这里就必须改成 memory_key="chat_history"，否则会报错！
    memory_key="history",
    # (进阶拓展：这里其实还有另外两个常用隐藏参数，虽然你没写，但很有用)
    #
    # return_messages=False (默认值)
    #   - 设为 False：记忆会以“纯字符串”格式返回，例如："Human: 你好\nAI: 你好！"（适合纯文本模板）
    #   - 设为 True：记忆会以“消息对象列表”格式返回，例如：[HumanMessage("你好"), AIMessage("你好！")]（适合 ChatPromptTemplate 模型）
    #
    # input_key 和 output_key
    #   - 当你的链有多个输入或输出时，用来明确告诉 memory 到底该把哪个字段存为“人类的话”，哪个字段存为“AI的话”。
)

# 2. 存储相关的消息
# inputs对应的就是用户信息，outputs对应的就是ai消息
memory.save_context(
    inputs={"human": "你好，我叫小明"},
    outputs={"ai": "很高兴认识你"},
)
memory.save_context(
    inputs={"input": "帮我谁打一下1+2*3=？"},
    outputs={"output": "7"},
)

# 3. 获取存储的信息
# 返回消息列表的方式1：
print(memory.load_memory_variables({}))

# 返回消息列表的方式2：
print(memory.chat_memory.messages)

# 说明：返回的字典结构的key叫history。

# 提供chain
chain = LLMChain(llm=chat_model, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})

# 此时history的内容为空
print(response)

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我谁打一下1+2*3=？\nAI: 7'}
[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我谁打一下1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
{'question': '你好，我的名字叫小明', 'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我谁打一下1+2*3=？\nAI: 7', 'text': '你好，小明！很高兴再次见到你。有什么我可以帮助你的吗？'}


In [10]:
# ConversationBufferMemory
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

template = """
你是一个非常幽默、说话接地气的人工智能助手。
你会根据之前的对话记忆来回答问题。如果遇到不懂的，你会用搞笑的方式承认。

=== 记忆档案 ===
{history}

=== 新情况 ===
人类：{input}
AI："""
# ==============================================================================
# ⚙️ 阶段 1：环境与基础底座配置
# ==============================================================================
# 加载环境变量 (如果你使用的是本地 Ollama，这两行防 502 报错的代理设置非常关键)
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

# 初始化大语言模型大脑 (这里以硅基流动的 Qwen 为例，你也可以换成 ChatOllama)
chat_model = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
)

# ==============================================================================
# 🌟 阶段 2：ConversationChain 全参数实例化
#
# 以下展示的，就是当你调用 ConversationChain 时，可以配置的所有核心参数。
# 如果你不写它们，LangChain 也会使用我在注释里标注的【默认值】。
# ==============================================================================
conv_chain = ConversationChain(
    # ------------------ 【必填参数】 ------------------
    # 🧠 1. llm (必需)
    # 作用：提供算力的“大脑”。这是唯一一个你必须手动传给它的参数。
    llm=chat_model,
    # ------------------ 【调试与监控】 ------------------
    # 🔍 2. verbose
    # 默认值：False
    # 作用：“啰嗦模式”。设为 True 后，控制台会用绿色字体打印出它拼接好的完整 Prompt。
    # 职场建议：开发调试时必开 (True)，上线给客户用时必关 (False)。
    verbose=True,
    # ------------------ 【被隐藏的“懒人”参数】 ------------------
    # 💾 3. memory
    # 默认值：ConversationBufferMemory()
    # 作用：指定使用哪个记忆管家。
    memory=ConversationBufferMemory(memory_key="history" \
    ""),
    # 📝 4. prompt
    # 默认值：LangChain 官方内置的一段英文闲聊模板。
    # 注意：如果你要传自定义模板，模板里必须包含 {history} 和 {input} 两个变量。
    prompt=PromptTemplate.from_template(template=template),
    # ------------------ 【输入输出定制参数】 ------------------
    # 📥 5. input_key
    # 默认值："input"
    # 作用：告诉这个链条，“当用户调用 invoke 的时候，字典里的哪个 Key 代表用户说的话？”
    input_key="input",
    # 📤 6. output_key
    # 默认值："response"
    # 作用：告诉这个链条，“当你算出结果后，把 AI 的回答存进字典的哪个 Key 里返回？”
    output_key="response",
    # ------------------ 【骨灰级高阶参数】 ------------------
    # 📡 7. callbacks
    # 默认值：None
    # 作用：回调函数列表，用于打字机流式输出或接驳云端监控（此处留空备用）。
    callbacks=[],
)

# ==============================================================================
# 🚀 阶段 3：发起多轮对话测试 (观察记忆的积累)
# ==============================================================================
print("\n" + "=" * 50)
print("💬 对话开始 (请注意观察上方绿色的 Verbose 调试信息)")
print("=" * 50)

# 第一轮对话：抛出设定
print("\n👨‍💻 人类: 你好，我叫小智，我最喜欢吃麻辣火锅。")
# 注意：这里的 "input" 必须和上面配置的 input_key="input" 保持一致
res1 = conv_chain.invoke({"input": "你好，我叫小智，我最喜欢吃麻辣火锅。"})
# 注意：获取答案的 "response" 必须和上面配置的 output_key="response" 保持一致
print(f"🤖 AI: {res1['response']}")

# 第二轮对话：测试记忆
print("\n👨‍💻 人类: 咱们去吃饭吧，你还记得我要吃什么吗？")
res2 = conv_chain.invoke({"input": "咱们去吃饭吧，你还记得我要吃什么吗？"})
print(f"🤖 AI: {res2['response']}")

# 第三轮对话：再次测试记忆叠加
print("\n👨‍💻 人类: 对了，千万别忘了我叫什么名字！")
res3 = conv_chain.invoke({"input": "对了，千万别忘了我叫什么名字！"})
print(f"🤖 AI: {res3['response']}")

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# ConversationBuffrtWindowMemory的使用
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import LLMChain

# ==============================================================================
# ⚙️ 阶段 1：环境与基础底座配置
# ==============================================================================
# 加载环境变量 (如果你使用的是本地 Ollama，这两行防 502 报错的代理设置非常关键)
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

# 初始化大语言模型大脑 (这里以硅基流动的 Qwen 为例，你也可以换成 ChatOllama)
chat_model = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
)

# ==============================================================================
# 📝 阶段 2：定制提示词模板 (PromptTemplate)
# ==============================================================================
template = """以下是人类与AI之间的友好对话描述。
AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会诚实地表示不知道。

=== 当前记忆档案 ===
{history}
====================

Human: {question}
AI:"""

prompt_template = PromptTemplate.from_template(template)

# ==============================================================================
# 🪟 阶段 3：实例化“滑动窗口”记忆管家
#
# 核心知识点剖析：
# - "Window (窗口)" 意味着它只会保留最近的几次对话，太老的对话会被直接丢弃。
# - 参数 k：代表保留的“对话轮数”。
#   ⚠️ 极度易错点：k=1 代表保留“1个用户问题 + 1个AI回答”，而不是1句话！
# ==============================================================================
memory = ConversationBufferWindowMemory(
    k=1,  # 🌟 核心参数：这里设为1，意味着 AI 永远只能记住“上一次”的对话。
    memory_key="history",  # 必须和模板里的 {history} 对应
    input_key="question",  # 推荐明确指定：告诉管家去抓取字典里的 "question" 作为用户的提问
)

# ==============================================================================
# 🔗 阶段 4：组装 LLMChain
# ==============================================================================
window_chain = LLMChain(
    llm=chat_model,
    prompt=prompt_template,
    memory=memory,
    verbose=True,  # 开启上帝视角，亲眼看看它是怎么“失忆”的
)

# ==============================================================================
# 🚀 阶段 5：发起对话测试 (见证 k=1 的“金鱼记忆”时刻)
# ==============================================================================
print("\n" + "=" * 50)
print("💬 滑动窗口对话开始 (当前设定 k=1)")
print("=" * 50)

print("\n--- 💬 第一轮：自报家门 ---")
res1 = window_chain.invoke({"question": "你好，我是孙小空。"})
print(res1)
print("🤖 AI:", res1["text"])
# 此时记忆库：[孙小空]

print("\n--- 💬 第二轮：介绍师弟 ---")
res2 = window_chain.invoke({"question": "我还有两个师弟，一个是猪小戒，一个是沙小僧。"})
print(res2)
print("🤖 AI:", res2["text"])
# 此时记忆库：[师弟] (🚨 注意：因为 k=1，"孙小空" 这个记忆已经被挤出窗口，永久丢失了！)

print("\n--- 💬 第三轮：聊聊高考 ---")
res3 = window_chain.invoke({"question": "我今年高考，竟然考上了1本！"})
print(res3)
print("🤖 AI:", res3["text"])
# 此时记忆库：[考上1本] (🚨 "师弟" 的记忆也被挤出去了！)

print("\n--- 💬 第四轮：终极灵魂拷问 ---")
res4 = window_chain.invoke({"question": "那你还记得我叫什么名字吗？我的师弟叫什么？"})
print(res4)
print("🤖 AI 最终回复:", res4["text"])
# 预期结果：AI 会诚实地表示不知道，因为它脑子里现在只有“你考上了1本”这件事。


💬 滑动窗口对话开始 (当前设定 k=1)

--- 💬 第一轮：自报家门 ---


> Entering new LLMChain chain...
Prompt after formatting:
以下是人类与AI之间的友好对话描述。
AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会诚实地表示不知道。

=== 当前记忆档案 ===


Human: 你好，我是孙小空。
AI:

> Finished chain.
{'question': '你好，我是孙小空。', 'history': '', 'text': '你好，孙小空！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我在这里可以帮你解答各种问题，提供信息和建议，甚至陪聊解闷。你今天想和我聊些什么呢？有什么我可以帮助你的吗？'}
🤖 AI: 你好，孙小空！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我在这里可以帮你解答各种问题，提供信息和建议，甚至陪聊解闷。你今天想和我聊些什么呢？有什么我可以帮助你的吗？

--- 💬 第二轮：介绍师弟 ---


> Entering new LLMChain chain...
Prompt after formatting:
以下是人类与AI之间的友好对话描述。
AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会诚实地表示不知道。

=== 当前记忆档案 ===
Human: 你好，我是孙小空。
AI: 你好，孙小空！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我在这里可以帮你解答各种问题，提供信息和建议，甚至陪聊解闷。你今天想和我聊些什么呢？有什么我可以帮助你的吗？

Human: 我还有两个师弟，一个是猪小戒，一个是沙小僧。
AI:

> Finished chain.
{'question': '我还有两个师弟，一个是猪小戒，一个是沙小僧。', 'history': 'Human: 你好，我是孙小空。\nAI: 你好，孙小空！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我在这里可以帮你解答各种问题，提供信息和建议，甚至陪聊解闷。你今天想和我聊些什么呢？有什么我可以帮助你的吗？', 'text': '你好，孙小空！很

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationTokenBufferMemory

# ==============================================================================
# ⚙️ 阶段 1：环境与基础底座配置
# ==============================================================================
dotenv.load_dotenv(override=True)

# 初始化大语言模型
# ⚠️ 注意：这里最好指定具体的模型名称，因为不同模型的 Token 计算公式不一样！
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
    # 🌟 救命金牌：专门用来解决 Token 计算器报错的参数！
    tiktoken_model_name="gpt-3.5-turbo",
)

# ==============================================================================
# 🧮 阶段 2：实例化“按 Token 计费”的记忆管家
#
# 核心差异剖析：
# - 之前的 WindowMemory 是按“轮数(k)”截断。它不管你一句话是 10 个字还是 10000 个字。
# - TokenBufferMemory 是按“底层字符切片(Token)”截断。它完美契合了 AI 真实的脑容量。
# ==============================================================================
memory = ConversationTokenBufferMemory(
    # 🌟 核心参数 1：为什么记忆管家需要传入大模型？
    # 因为管家自己并不知道一句话到底占几个 Token。它必须借助这个 llm 底层的 Tokenizer（分词器）
    # 来精确“称重”，算清楚每句话多重，才能决定要不要丢弃老记忆。
    llm=llm,
    # 🌟 核心参数 2：设定的“脑容量”上限（Token 数）
    # 这里设为 50，意味着只要对话总长度超过 50 个 Token，最老的那句话就会被强制踢出去。
    # token上限的默认值为2000
    max_token_limit=55,
    # 依然保留接头暗号（供后续接入 LLMChain 使用）
    memory_key="history",
)

# ==============================================================================
# 💾 阶段 3：模拟多轮对话的疯狂注入
# ==============================================================================
print("\n" + "=" * 55)
print("⏳ 正在向容量为 50 Token 的大脑注入记忆...")
print("=" * 55)

# 第一句话（比较短）
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
print(memory.chat_memory.messages)
# 第二句话（比较长，可能会触发“容量报警”）
memory.save_context(
    {"input": "今天天气如何？我想出去打篮球。"},
    {"output": "今天是晴天，气温25度，非常适合户外运动，但记得涂防晒霜哦！"},
)

# ==============================================================================
# 🔍 阶段 4：开箱验货 (检查谁被挤出去了)
# ==============================================================================
print("\n🔍 【透视 Memory 内部残留数据】:")
print(memory.chat_memory.messages)
# 查看当前还在管家肚子里的记忆
current_memory = memory.load_memory_variables({})["history"]
print(current_memory)

print("\n💡 结论：你可以观察到，如果上面两轮对话的总 Token 超过了 50，")
print("最老的『你好吗？』那轮对话就会被部分或全部丢弃，只留下最新的天气记录！，如果超过了token容量的限制，不会完整的丢弃一对，但是会丢弃完整的一部分如丢弃一整个input")


⏳ 正在向容量为 50 Token 的大脑注入记忆...
[HumanMessage(content='你好吗？'), AIMessage(content='我很好，谢谢！')]

🔍 【透视 Memory 内部残留数据】:
[AIMessage(content='今天是晴天，气温25度，非常适合户外运动，但记得涂防晒霜哦！')]
AI: 今天是晴天，气温25度，非常适合户外运动，但记得涂防晒霜哦！

💡 结论：你可以观察到，如果上面两轮对话的总 Token 超过了 50，
最老的『你好吗？』那轮对话就会被部分或全部丢弃，只留下最新的天气记录！


In [ ]:
# ConversationSummaryMemory
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationSummaryMemory

# 🌟 规范避坑：在现代 LangChain 中，推荐从 core 导入 InMemoryChatMessageHistory 作为底层历史记录载体
from langchain_core.chat_history import InMemoryChatMessageHistory

# ==============================================================================
# ⚙️ 阶段 1：环境与基础底座配置
# ==============================================================================
dotenv.load_dotenv(override=True)

# 初始化大语言模型
# ⚠️ 注意：这里最好指定具体的模型名称，因为不同模型的 Token 计算公式不一样！
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
    # 🌟 救命金牌：专门用来解决 Token 计算器报错的参数！
    tiktoken_model_name="gpt-3.5-turbo",
)

# ==============================================================================
# 📚 阶段 2：准备“又长又啰嗦”的原始聊天记录 (充当外部物理硬盘)
# ==============================================================================
# 我们先创建一个纯粹用来装聊天记录的容器（相当于没经过压缩的原始录音）
history = InMemoryChatMessageHistory()

# 故意塞入一堆啰嗦的对话，用来测试摘要功能
history.add_user_message("你好，我是孙小空，我来自花果山。")
history.add_ai_message("你好，孙小空！花果山是个好地方。")
history.add_user_message(
    "我师父叫唐三藏，我还有两个师弟，猪小戒和沙小僧。我们打算去西天取经。"
)
history.add_ai_message("哇，这是一个伟大的团队！祝你们取经顺利。")
history.add_user_message("不过我最喜欢吃的是水蜜桃，千万别给我吃香蕉。")
history.add_ai_message("记住了，你只爱吃水蜜桃，不吃香蕉！")

print("\n" + "=" * 50)
print("⏳ 正在唤醒大模型，对原始冗长对话进行【高能压缩总结】...")
print("=" * 50)

# ==============================================================================
# 🗜️ 阶段 3：全参数实例化 ConversationSummaryMemory
#
# 我们使用的是 .from_messages() 这个高级工厂方法。
# 它会在初始化的瞬间，立刻消耗一次 LLM 算力，把上面 history 里的长对话读一遍，并写出摘要。
# ==============================================================================
memory = ConversationSummaryMemory.from_messages(
    # ------------------ 【必填核心参数】 ------------------
    # 🧠 1. llm (必需)
    # 作用：充当“速记员”。管家自己不会写摘要，它必须调用这个大模型来帮它写小作文。
    llm=llm,
    # 💾 2. chat_memory (必需，前提是你使用 from_messages 方法)
    # 作用：告诉管家，你要压缩的“原始数据源”在哪里？把它传进来。
    chat_memory=history,
    # ------------------ 【高阶/定制参数】 ------------------
    # 🔄 3. return_messages
    # 默认值：False
    # 作用：控制最终摘要输出的格式。
    #  - 设为 False：返回一段纯文本，例如："用户叫孙小空，爱吃水蜜桃..." (适合普通的 PromptTemplate)
    #  - 设为 True 🌟：返回一个 SystemMessage 对象！大模型会认为这是系统赋予它的背景设定，服从度极高！(适合 ChatPromptTemplate)
    return_messages=True,
    # 🔑 4. memory_key
    # 默认值："history"
    # 作用：接头暗号。以后放进 Prompt 模板里供 LLMChain 填空用的占位符名字。
    memory_key="history",
    # 📝 5. prompt (隐藏参数，无需修改，但必须知道底层的存在)
    # 默认值：LangChain 内置的一段英文要求：“请逐步总结以下提供的对话，并将其添加到之前的摘要中...”
    # 进阶作用：你甚至可以替换掉官方的总结要求，让它用特定的口吻（比如文言文，或者只提取 JSON 格式）来写摘要。
    # prompt=自定义的PromptTemplate
)

# ==============================================================================
# 🔍 阶段 4：开箱验货 (见证奇迹的时刻)
# ==============================================================================
print("\n📦 【透视：压缩后的记忆数据 (将喂给大模型)】:")
# load_memory_variables() 是最终传给链 (Chain) 的数据提取动作
compressed_memory = memory.load_memory_variables({})
print(compressed_memory)

# 你也可以直接查看纯文本摘要结果
print("\n📝 【提取底层的纯文本摘要 (buffer 属性)】:")
print(memory.buffer)

In [ ]:
# ConversationSummayBufferMemory
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory

# ==============================================================================
# ⚙️ 阶段 1：环境与基础底座配置
# ==============================================================================
dotenv.load_dotenv(override=True)

# 初始化大语言模型
# ⚠️ 注意：这里最好指定具体的模型名称，因为不同模型的 Token 计算公式不一样！
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
    # 🌟 救命金牌：专门用来解决 Token 计算器报错的参数！
    tiktoken_model_name="gpt-3.5-turbo",
)

# ==============================================================================
# 🗜️ 阶段 2：全参数实例化 ConversationSummaryBufferMemory
#
# 核心原理：它肚子分为两部分 => [摘要区] + [原话缓冲池]
# ==============================================================================
memory_limit_40 = ConversationSummaryBufferMemory(
    # ------------------ 【必填核心参数】 ------------------
    # 🧠 1. llm (必需)
    # 作用：充当“摘要生成员”。当缓冲池溢出时，调用它来把溢出的旧对话压缩成摘要。
    llm=llm,
    # ⚖️ 2. max_token_limit (必需)
    # 作用：缓冲池的“警戒线”（Token 数量）。
    # 机制：只要最近对话的总 Token 没超过这个值，它就原样保存。一旦超过，
    #       最老的那句话不会被丢弃，而是被抽出来发给 LLM 写进摘要里！
    max_token_limit=40,
    # ------------------ 【高阶/定制参数】 ------------------
    # 🔄 3. return_messages
    # 默认值：False
    # 作用：决定你最后拿到的数据长什么样。
    # 设为 True 🌟：返回对象列表。摘要会变成 SystemMessage，原话保留为 Human/AIMessage。
    return_messages=True,
    # 🔑 4. memory_key
    # 默认值："history"
    # 作用：接头暗号，给 Prompt 模板预留的变量名。
    memory_key="history",
    # 📥 5. input_key / output_key (可选)
    # 作用：如果传入的字典有多个键值对，明确告诉管家哪个是人类的输入，哪个是AI的输出。
    # input_key="input",
    # output_key="output"
)

# 为了做对比，我们再建一个阈值极低（20）的管家
memory_limit_20 = ConversationSummaryBufferMemory(
    llm=llm, max_token_limit=20, return_messages=True
)


# ==============================================================================
# 💾 阶段 3：注入一模一样的历史消息
# ==============================================================================
def inject_messages(memory_obj):
    memory_obj.save_context(
        {"input": "你好，请帮问一下康师傅明天是否有空"}, {"output": "您好，康师傅是谁"}
    )
    memory_obj.save_context(
        {"input": "康师傅是IT培训机构尚硅谷的讲师"}, {"output": "好的，我知道了"}
    )
    memory_obj.save_context(
        {"input": "希望明天他来帮我规划一下职业生涯"}, {"output": "好的，我来告诉他"}
    )


print("⏳ 正在为【40 Token 管家】注入记忆...")
inject_messages(memory_limit_40)

print("⏳ 正在为【20 Token 管家】注入记忆...")
inject_messages(memory_limit_20)

# ==============================================================================
# 🔍 阶段 4：开箱验货 (见证“原话”到“摘要”的奇妙转化)
# ==============================================================================
print("\n" + "=" * 50)
print("📦 【透视：max_token_limit=40 的记忆结构】")
print("因为 40 Token 比较宽裕，你应该能看到更多的原话被保留：")
print(memory_limit_40.load_memory_variables({})["history"])

print("\n" + "=" * 50)
print("📦 【透视：max_token_limit=20 的记忆结构】")
print(
    "因为 20 Token 极小，几乎存不下一句原话。前面的对话一定被强制压缩成了 SystemMessage："
)
print(memory_limit_20.load_memory_variables({})["history"])

⏳ 正在为【40 Token 管家】注入记忆...
⏳ 正在为【20 Token 管家】注入记忆...

📦 【透视：max_token_limit=40 的记忆结构】
因为 40 Token 比较宽裕，你应该能看到更多的原话被保留：
[SystemMessage(content='The human greets the AI and asks if they can help check if Kangshifu is available tomorrow. The AI responds by asking who Kangshifu is, and the human clarifies that Kangshifu is an IT training instructor at Shanggugu. The AI acknowledges the information and the human expresses the hope that Kangshifu will come tomorrow to help plan their career.'), AIMessage(content='好的，我来告诉他')]

📦 【透视：max_token_limit=20 的记忆结构】
因为 20 Token 极小，几乎存不下一句原话。前面的对话一定被强制压缩成了 SystemMessage：
[SystemMessage(content='The human greets the AI and asks if they can help inquire about whether Kangshifu is available tomorrow. The AI responds by asking who Kangshifu is, and the human clarifies that Kangshifu is a lecturer at the IT training institution Shangguguan. The AI acknowledges the information and the human expresses the hope that Kangshifu will be available tomorrow to help

In [ ]:
# Memory的长度限制
from langchain_core.prompts import PromptTemplate
from langchain.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI

# 1. 准备你的大模型
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==============================================================================
# 🌟 核心杀招：自定义摘要提示词 (Custom Summary Prompt)
#
# ⚠️ 铁律：模板中必须包含 {summary}（旧摘要）和 {new_lines}（新对话）这两个变量！
# ==============================================================================
strict_summary_template = """
你是一个极其精简的速记员。请将以下对话进行极致的压缩总结。

【严格要求】：
1. 必须用中文回答。
2. 绝对不能超过 50 个字！
3. 只保留用户的人设、核心诉求等关键事实，剔除所有客套话。
4. 如果内容过长，请使用条目式的短语（如：用户叫张三；想学Python）。

当前已有摘要：
{summary}

新增的对话记录：
{new_lines}

请输出最新的极致压缩摘要：
"""

# 将字符串转为 PromptTemplate 对象
custom_prompt = PromptTemplate.from_template(strict_summary_template)

# ==============================================================================
# 🗜️ 实例化管家，并注入你的“紧箍咒”提示词
# ==============================================================================
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=40,
    return_messages=True,
    # 🌟 就在这里！把官方啰嗦的提示词替换成你写的严苛版
    prompt=custom_prompt,
)

# 测试一下
memory.save_context(
    {"input": "你好，请帮问一下康师傅明天是否有空"}, {"output": "您好，康师傅是谁"}
)
memory.save_context(
    {"input": "康师傅是IT培训机构尚硅谷的讲师"}, {"output": "好的，我知道了"}
)

# 查看结果，你会发现大模型变得非常乖巧，摘要字数被死死按住了！
print(memory.load_memory_variables({})["history"])

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

# 1. 准备大模型（同样建议 temperature=0 保证摘要的严谨性）
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==============================================================================
# 🌟 核心杀招：自定义极简版摘要提示词
# 
# 再次划重点：无论你提示词怎么改，必须留下 {summary} 和 {new_lines} 这两个坑位！
# 否则 LangChain 底层在填空时会因为找不到变量而报错。
# ==============================================================================
strict_prompt_template = """
你是一个极其没有耐心的速记员。请将以下对话进行极致的压缩总结。

【最高指令】：
1. 绝对不能超过 30 个汉字！
2. 像发文言文电报一样，只提取核心名词和意图。

当前已有的摘要档案：
{summary}

刚刚新增的废话对话：
{new_lines}

请输出最新的极致压缩版摘要：
"""

custom_prompt = PromptTemplate.from_template(strict_prompt_template)

# ==============================================================================
# 🗜️ 实例化 ConversationSummaryMemory，并强行注入你的提示词
# ==============================================================================
memory = ConversationSummaryMemory(
    llm=llm, 
    return_messages=True, # 依然推荐开启，把它变成系统消息
    
    # 🌟 就在这里！用你写的严苛版，干掉官方默认的啰嗦版
    prompt=custom_prompt 
)

# 测试一下效果
memory.save_context({"input": "你好，我是孙小空，来自花果山。我师父叫唐三藏，我师弟叫猪小戒。"}, {"output": "你好孙小空！你的团队很壮大！"})
memory.save_context({"input": "对啊，我们打算明天就出发去西天取经了。"}, {"output": "祝你们一路顺风！"})

# 盲猜一下，大模型肯定不敢多写一个字！
print(memory.load_memory_variables({})['history'])
